# Module 7 Day 1 — TaskCompletionMetric + ToolCorrectnessMetric

**Module 7 · Session 1 of 2**

Module 6 gave you a multi-hop retrieval agent and three hand-rolled verdict functions. Today we:
1. Upgrade the agent to **real tool-calling** (OpenAI function-calling API)
2. Replace `GracefulFailureVerdict` with `TaskCompletionMetric`
3. Replace `ReasoningChainVerdict` with `ToolCorrectnessMetric`
4. Apply **Module 4 Day 4 equivalence partitioning** to tool selection
5. Build hard negatives for the **Air Canada pattern** (no tool called)

---

**The anchor incident:** In February 2024, Air Canada's AI chatbot told a grieving passenger that bereavement fares could be claimed *retroactively* — a policy that didn't exist. Air Canada was found liable by Canada's Civil Resolution Tribunal. The chatbot's failure was specifically that it **answered a policy question without calling the authoritative tool** (`get_product_info`). That's what `ToolCorrectnessMetric` catches.

## Setup

In [1]:
import asyncio
import json
import os
import sys
from pathlib import Path

# agent_tools.py is in the same directory as this notebook
sys.path.insert(0, str(Path(".").resolve()))

from agent_tools import (
    run_agent,
    to_deepeval_tool_calls,
    check_order_status,
    get_product_info,
    ORDERS,
    PRODUCTS,
)

provider = os.getenv("PROVIDER", "azure")
print(f"PROVIDER={provider}")

tracing = os.getenv("LANGSMITH_TRACING", "false").lower() == "true"
project = os.getenv("LANGSMITH_PROJECT", "module-07-deepeval-agent-testing")
if tracing:
    print(f"LangSmith tracing enabled — project: {project}")
else:
    print("LangSmith tracing disabled (set LANGSMITH_TRACING=true to enable)")

PROVIDER=azure
LangSmith tracing enabled — project: module-07-deepeval-agent-testing


## Part 1 — The in-memory database and tools

Quick inspection of what the agent knows about.

In [2]:
print("ORDERS database:")
for oid, o in ORDERS.items():
    print(f"  {oid}: {o['status']} — {o['product']}, arrives in {o['estimated_delivery']}, ${o['amount']:.2f}")

print()
print("PRODUCTS catalogue:")
for pname, p in PRODUCTS.items():
    print(f"  {pname}: {p['price']}, cancellation_fee={p['cancellation_fee']}, support={p['support_type']}")

print()
print("Note: TurboMax Pro cancellation_fee='unknown' is intentional.")
print("A correct agent should report this honestly — not invent a number.")
print("This is the Air Canada pattern if the agent answers without calling get_product_info.")

ORDERS database:
  12345: shipped — WidgetPro 3000, arrives in 2 days, $199.00
  67890: processing — TurboMax Pro, arrives in 5 days, $299.00
  99999: delivered — WidgetPro 3000, no ETA, $199.00

PRODUCTS catalogue:
  WidgetPro 3000: $199, cancellation_fee=$0, support=online
  TurboMax Pro: $299/year, cancellation_fee=unknown, support=online
  WidgetPro 2000: discontinued, cancellation_fee=$50, support=none

Note: TurboMax Pro cancellation_fee='unknown' is intentional.
A correct agent should report this honestly — not invent a number.
This is the Air Canada pattern if the agent answers without calling get_product_info.


## Part 2 — Run the agent on a happy-path case

The agent uses OpenAI's `tools` parameter. With `verbose=True`, you can see the exact tool call the LLM made — including the function name and arguments it chose.

**LangSmith note:** with tracing enabled, this call appears as a `customer_service_agent` root span with a nested `check_order_status` child span (run_type="tool"). You can inspect the raw arguments and output per-span in the LangSmith UI — same `@traceable` pattern from Modules 5 and 6, now applied to individual tool calls.

In [3]:
response = asyncio.run(
    run_agent("What's the status of my order #12345?", verbose=True)
)

print("Agent output:")
print(f"  {response.output}")
print()
print("Tool calls made:")
for i, tc in enumerate(response.tools_called, 1):
    print(f"  {i}. {tc.name}({', '.join(f'{k}={v!r}' for k, v in tc.input_parameters.items())})")
    print(f"     → {tc.output}")

[tool call] check_order_status({'order_id': '12345'})
[tool result] {'status': 'shipped', 'product': 'WidgetPro 3000', 'estimated_delivery': '2 days', 'amount': 199.0}

Agent output:
  Your order #12345 (WidgetPro 3000) has shipped! It's estimated to arrive within 2 days. Is there anything else I can help you with?

Tool calls made:
  1. check_order_status(order_id='12345')
     → {"status": "shipped", "product": "WidgetPro 3000", "estimated_delivery": "2 days", "amount": 199.0}


## Part 3 — Equivalence partitioning: tool selection (Module 4 Day 4 applied)

Module 4 Day 4 introduced equivalence partitioning for input space. Today we apply the same technique to the **tool selection axis** — not the input value axis.

Without this partition, a test suite built entirely from "correct tool, correct args" cases will pass 100% and catch nothing.

In [4]:
# Module 4 Day 4: equivalence partitioning applied to tool selection
tool_selection_partitions = {
    "correct_tool_correct_args": {
        "description": "happy path — right tool, right input",
        "example": "order #12345 → check_order_status('12345')",
        "coverage": ["order-status-01", "refund-01", "product-info-01", "escalation-01"],
    },
    "wrong_tool": {
        "description": "process_refund called for a status check",
        "example": "order status query → process_refund('12345', 'status inquiry')",
        "coverage": ["order-status-01-hardneg (Day 1)"],
    },
    "no_tool_called": {
        "description": "LLM answers from parametric memory instead of calling a tool",
        "example": "TurboMax Pro cancellation fee → invented $25 answer, tools_called=[]",
        "coverage": ["product-info-01-hardneg (Day 1) — the Air Canada pattern"],
    },
    "correct_tool_wrong_args": {
        "description": "check_order_status called with wrong order_id (contaminated from context)",
        "example": "user asks about #67890, agent passes order_id='12345'",
        "coverage": ["refund-01-hardneg (Day 2 — ArgumentCorrectnessMetric)"],
    },
}

print("Tool selection equivalence partitions (Module 4 Day 4 technique applied to a new axis):")
for partition, info in tool_selection_partitions.items():
    print(f"\n  PARTITION: {partition}")
    print(f"  Description: {info['description']}")
    print(f"  Example: {info['example']}")
    print(f"  Coverage: {', '.join(info['coverage'])}")

print()
print("Coverage matrix — which cases cover which cells:")
print("+-----------------------------+---------------------+-----------------------+------------------+----------------------------+")
print("| Capability                  | correct_tool_args   | wrong_tool            | no_tool_called   | correct_tool_wrong_args    |")
print("+-----------------------------+---------------------+-----------------------+------------------+----------------------------+")
print("| order_management            | order-status-01     | order-status-01-HN    | —                | refund-01-HN (Day 2)       |")
print("| product_inquiry             | product-info-01     | —                     | product-info-HN  | —                          |")
print("| refund_processing           | refund-01           | —                     | —                | refund-01-HN (Day 2)       |")
print("| escalation                  | escalation-01       | —                     | —                | —                          |")
print("+-----------------------------+---------------------+-----------------------+------------------+----------------------------+")
print()
print("Gaps visible: '—' cells are your next test-writing task.")

Tool selection equivalence partitions (Module 4 Day 4 technique applied to a new axis):

  PARTITION: correct_tool_correct_args
  Description: happy path — right tool, right input
  Example: order #12345 → check_order_status('12345')
  Coverage: order-status-01, refund-01, product-info-01, escalation-01

  PARTITION: wrong_tool
  Description: process_refund called for a status check
  Example: order status query → process_refund('12345', 'status inquiry')
  Coverage: order-status-01-hardneg (Day 1)

  PARTITION: no_tool_called
  Description: LLM answers from parametric memory instead of calling a tool
  Example: TurboMax Pro cancellation fee → invented $25 answer, tools_called=[]
  Coverage: product-info-01-hardneg (Day 1) — the Air Canada pattern

  PARTITION: correct_tool_wrong_args
  Description: check_order_status called with wrong order_id (contaminated from context)
  Example: user asks about #67890, agent passes order_id='12345'
  Coverage: refund-01-hardneg (Day 2 — ArgumentCor

## Part 4 — DeepEval: TaskCompletionMetric

We map the `GracefulFailureVerdict` from Module 6 to `TaskCompletionMetric` here. Instead of checking for hedge phrases in a string, DeepEval uses an LLM judge to ask: "Given the task description and this output, was the task completed?"

In [5]:
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import TaskCompletionMetric

# Happy-path case: correct tool called, task completed
happy_path_case = LLMTestCase(
    input="What's the status of my order #12345?",
    actual_output="Your order #12345 (WidgetPro 3000) has shipped! It's estimated to arrive within 2 days.",
    tools_called=[
        ToolCall(
            name="check_order_status",
            input_parameters={"order_id": "12345"},
            output='{"status": "shipped", "product": "WidgetPro 3000", "estimated_delivery": "2 days", "amount": 199.0}',
        )
    ],
)

task_metric = TaskCompletionMetric(
    task="Help customers with order status, refunds, and product questions",
    threshold=0.7,
    verbose_mode=True,
)

print("Running TaskCompletionMetric on the happy-path order status case...")
task_metric.measure(happy_path_case)
print()
print("TaskCompletionMetric result:")
print(f"  Score:       {task_metric.score:.2f}")
print(f"  Passed:      {task_metric.is_successful()}  (threshold=0.7)")
print(f"  Reason:      {task_metric.reason}")

Running TaskCompletionMetric on the happy-path order status case...

TaskCompletionMetric result:
  Score:       0.92
  Passed:      True  (threshold=0.7)
  Reason:      The agent correctly identified the order status (shipped) and provided the estimated delivery time (2 days). The customer's question was fully answered with accurate information retrieved from the order system.


## Part 5 — DeepEval: ToolCorrectnessMetric

`ToolCorrectnessMetric` inspects `tools_called` — not just the final output text. It answers: "Were the right tools called for this task?"

This is what `ReasoningChainVerdict` from Module 6 was approximating with string matching.

In [6]:
from deepeval.metrics import ToolCorrectnessMetric

tool_metric = ToolCorrectnessMetric(threshold=0.7, verbose_mode=True)

# Happy path
tool_metric.measure(happy_path_case)
print("ToolCorrectnessMetric on happy-path case:")
print(f"  Score: {tool_metric.score:.2f} | Passed: {tool_metric.is_successful()}")
print(f"  Reason: {tool_metric.reason}")
print()

# Hard negative: wrong tool (process_refund for a status check)
wrong_tool_case = LLMTestCase(
    input="What's the status of my order #12345?",
    actual_output="I've processed a refund for order #12345 in the amount of $199.00.",
    tools_called=[
        ToolCall(
            name="process_refund",  # WRONG — user asked for status, not refund
            input_parameters={"order_id": "12345", "reason": "status inquiry"},
            output='{"refund_id": "R-ABCD12", "amount": 199.0, "status": "initiated", "reason": "status inquiry"}',
        )
    ],
)

tool_metric.measure(wrong_tool_case)
print("ToolCorrectnessMetric on WRONG TOOL case (process_refund for status check):")
print(f"  Score: {tool_metric.score:.2f} | Passed: {tool_metric.is_successful()}")
print(f"  Reason: {tool_metric.reason}")
print()
print(f">>> WRONG_TOOL correctly {'FAILED' if not tool_metric.is_successful() else 'PASSED (unexpected!)'} ToolCorrectnessMetric. This is the expected behavior.")

ToolCorrectnessMetric on happy-path case:
  Score: 0.95 | Passed: True
  Reason: check_order_status was the appropriate tool for an order status query and was called correctly.

ToolCorrectnessMetric on WRONG TOOL case (process_refund for status check):
  Score: 0.10 | Passed: False
  Reason: The agent called process_refund when the user only asked for order status. This is incorrect — check_order_status should have been called. Initiating a refund for a status inquiry is actively harmful to the customer.

>>> WRONG_TOOL correctly FAILED ToolCorrectnessMetric. This is the expected behavior.


## Part 6 — Hard negative: the Air Canada pattern (`tools_called=[]`)

The most dangerous case: the agent answers a policy question without calling any tool. The output may sound plausible, but it bypassed the authoritative data source.

**Key observation:** `TaskCompletionMetric` alone may PASS this case (the answer *sounds* complete). Only `ToolCorrectnessMetric` catches it reliably because it sees `tools_called=[]`.

In [7]:
# The Air Canada pattern: answer from memory, no tool called
no_tool_case = LLMTestCase(
    input="What's the cancellation fee for TurboMax Pro?",
    # Hallucinated — TurboMax Pro's cancellation_fee is 'unknown' in the real DB
    actual_output="The cancellation fee for TurboMax Pro is $25 if you cancel within the first 30 days, and $50 after that.",
    tools_called=[],  # EMPTY — no tool was called at all
)

print("Running both metrics on the 'no tool called' hard negative...")
print()

task_metric_fresh = TaskCompletionMetric(
    task="Help customers with order status, refunds, and product questions",
    threshold=0.7,
)
task_metric_fresh.measure(no_tool_case)
print("TaskCompletionMetric:")
print(f"  Score: {task_metric_fresh.score:.2f} | Passed: {task_metric_fresh.is_successful()}")
print(f"  Reason: {task_metric_fresh.reason}")
print()

tool_metric_fresh = ToolCorrectnessMetric(threshold=0.7)
tool_metric_fresh.measure(no_tool_case)
print("ToolCorrectnessMetric:")
print(f"  Score: {tool_metric_fresh.score:.2f} | Passed: {tool_metric_fresh.is_successful()}")
print(f"  Reason: {tool_metric_fresh.reason}")
print()

both_failed = not task_metric_fresh.is_successful() and not tool_metric_fresh.is_successful()
print(f"Both metrics correctly {'FAILED' if both_failed else 'PASSED — review results above'} on this hard negative.")
print()
print("Important: In some cases, TaskCompletionMetric might PASS a plausible-sounding hallucination.")
print("ToolCorrectnessMetric catches it regardless because tools_called=[] for a tool-dependent query.")
print("This is why you need BOTH metrics.")

Running both metrics on the 'no tool called' hard negative...

TaskCompletionMetric:
  Score: 0.55 | Passed: False
  Reason: While the agent provided a response about cancellation fees, the information provided ($25/$50) appears to be hallucinated rather than retrieved from the authoritative product system. The task requires accurate information.

ToolCorrectnessMetric:
  Score: 0.05 | Passed: False
  Reason: No tools were called. For a product policy question, get_product_info should have been invoked to retrieve authoritative data. The agent answered from memory, bypassing the tool that would have given the correct (or honest 'unknown') response.

Both metrics correctly FAILED on this hard negative.

Important: In some cases, TaskCompletionMetric might PASS a plausible-sounding hallucination.
ToolCorrectnessMetric catches it regardless because tools_called=[] for a tool-dependent query.
This is why you need BOTH metrics.


## Part 7 — Load hard negatives from golden_dataset.json and run DeepEval

The `golden_dataset.json` hard negatives for Day 1 are `order-status-01-hardneg` (wrong tool) and `product-info-01-hardneg` (no tool called). Let's load them and run the metrics.

In [8]:
with open("golden_dataset.json") as f:
    dataset = json.load(f)

# Day 1 hard negatives: tool_correctness eval_type
day1_hard_negatives = [
    d for d in dataset
    if d["is_hard_negative"] and d["eval_type"] == "tool_correctness"
]

print("Day 1 hard negatives from golden_dataset.json:")
for i, entry in enumerate(day1_hard_negatives, 1):
    tools = entry.get("tools_called", [])
    tool_desc = f"{tools[0]['name']} — WRONG" if tools else "NONE (tools_called=[])"
    if tools:
        if entry["id"] == "order-status-01-hardneg":
            tool_desc += " (should be check_order_status)"
    print(f"  {i}. {entry['id']} (eval_type={entry['eval_type']})")
    print(f"     Input: {entry['user_input']}")
    print(f"     Tool called: {tool_desc}")
    print(f"     Note: {entry.get('_note', 'N/A')[:75]}...")
    print()

Day 1 hard negatives from golden_dataset.json:
  1. order-status-01-hardneg (eval_type=tool_correctness)
     Input: What's the status of my order #12345?
     Tool called: process_refund — WRONG (should be check_order_status)
     Note: Wrong tool: the user asked for order STATUS, not a refund...

  2. product-info-01-hardneg (eval_type=tool_correctness)
     Input: What's the cancellation fee for TurboMax Pro?
     Tool called: NONE (tools_called=[])
     Note: No tool was called — the agent answered from parametric (training) mem...



In [9]:
def dataset_entry_to_test_case(entry: dict) -> LLMTestCase:
    """Convert a golden_dataset.json entry to a DeepEval LLMTestCase."""
    tools_called = [
        ToolCall(
            name=tc["name"],
            input_parameters=tc["input_parameters"],
            output=json.dumps(tc["output"]),
        )
        for tc in entry.get("tools_called", [])
    ]
    return LLMTestCase(
        input=entry["user_input"],
        actual_output=entry.get("response", ""),
        tools_called=tools_called,
    )

print("Running ToolCorrectnessMetric on Day 1 hard negatives...")
print()
all_correctly_failed = True
for entry in day1_hard_negatives:
    case = dataset_entry_to_test_case(entry)
    m = ToolCorrectnessMetric(threshold=0.7)
    m.measure(case)
    expected_fail = entry["is_hard_negative"]
    actually_failed = not m.is_successful()
    status = "OK" if (expected_fail == actually_failed) else "UNEXPECTED"
    print(f"  {entry['id']}:")
    print(f"    Score: {m.score:.2f}  Passed: {m.is_successful()}  (expected: {not expected_fail})")
    print(f"    {status} — metric {'correctly failed' if actually_failed else 'unexpectedly passed'} this hard negative.")
    print()
    if not actually_failed:
        all_correctly_failed = False

if all_correctly_failed:
    print("Both Day 1 hard negatives correctly trigger ToolCorrectnessMetric failure.")
else:
    print("WARNING: one or more hard negatives did not trigger failure — review above.")
print("If a hard negative PASSES, it means either: (1) metric threshold needs adjustment,")
print("or (2) the hard negative isn't actually hard enough — revise it.")

Running ToolCorrectnessMetric on Day 1 hard negatives...

  order-status-01-hardneg:
    Score: 0.08  Passed: False  (expected: False)
    OK — metric correctly failed this hard negative.

  product-info-01-hardneg:
    Score: 0.05  Passed: False  (expected: False)
    OK — metric correctly failed this hard negative.

Both Day 1 hard negatives correctly trigger ToolCorrectnessMetric failure.
If a hard negative PASSES, it means either: (1) metric threshold needs adjustment,
or (2) the hard negative isn't actually hard enough — revise it.


## Part 8 — Run the live agent on the Air Canada scenario

Force the agent to answer a TurboMax Pro cancellation question. The CORRECT behavior is calling `get_product_info` and reporting the 'unknown' fee honestly. If the live agent hallucinates instead, you'll see `tools_called=[]` — same as the hard negative above.

In [10]:
print("Live agent run — TurboMax Pro cancellation fee")
live_response = asyncio.run(
    run_agent("What's the cancellation fee for TurboMax Pro?", verbose=True)
)

print("Agent output:")
print(f"  {live_response.output}")
print()

if live_response.tools_called:
    print(f"Tool called: {live_response.tools_called[0].name}  (CORRECT — called the right tool)")
    print(f"Output included 'unknown': {'unknown' in live_response.output.lower()}  (CORRECT — reported the honest answer, not a fabricated fee)")
else:
    print("WARNING: No tool was called! Air Canada pattern detected — agent answered from memory.")

print()
print("Now running ToolCorrectnessMetric on this live response...")
live_case = LLMTestCase(
    input="What's the cancellation fee for TurboMax Pro?",
    actual_output=live_response.output,
    tools_called=to_deepeval_tool_calls(live_response),
)
live_metric = ToolCorrectnessMetric(threshold=0.7)
live_metric.measure(live_case)
print(f"  Score: {live_metric.score:.2f} | Passed: {live_metric.is_successful()}")
passed_str = "passed" if live_metric.is_successful() else "FAILED"
print(f"  The live agent {passed_str} ToolCorrectnessMetric{' — it called get_product_info as expected.' if live_metric.is_successful() else ' — review the tool calls above.'}")

Live agent run — TurboMax Pro cancellation fee
[tool call] get_product_info({'product_name': 'TurboMax Pro'})
[tool result] {'price': '$299/year', 'cancellation_fee': 'unknown', 'support_type': 'online'}

Agent output:
  I checked our product catalogue for TurboMax Pro. Unfortunately, the cancellation fee information is currently listed as 'unknown' in our system. I'd recommend reaching out to our online support team for the most accurate cancellation policy.

Tool called: get_product_info  (CORRECT — called the right tool)
Output included 'unknown': True  (CORRECT — reported the honest answer, not a fabricated fee)

Now running ToolCorrectnessMetric on this live response...
  Score: 0.93 | Passed: True
  The live agent passed ToolCorrectnessMetric — it called get_product_info as expected.


## Summary — Day 1

| What changed from Module 6 | Module 6 | Module 7 Day 1 |
|---|---|---|
| Agent type | Retrieval loop (decides when to fetch) | Tool-calling (decides which function + args) |
| GracefulFailureVerdict | Hedge-phrase string matching | `TaskCompletionMetric` (LLM-judged) |
| ReasoningChainVerdict | `must_include`/`must_not_include` checks | `ToolCorrectnessMetric` (inspects ToolCall records) |
| Coverage technique | Module 4 Day 4 partitions applied to hops | Module 4 Day 4 partitions applied to tool selection |

**Day 2:** `ArgumentCorrectnessMetric` (right tool, wrong order_id) + `StepEfficiencyMetric` (too many steps) + full coverage matrix.